# 05 — Ensemble Methods

Side-by-side comparison of sklearn vs rustml Random Forest and Gradient Boosting.

In [ ]:
import numpy as np
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, mean_squared_error
import matplotlib.pyplot as plt

## Random Forest Classifier

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=3, random_state=42)
X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400].astype(float), y[400:].astype(float)

for n_est in [10, 50, 100]:
    clf = RandomForestClassifier(n_estimators=n_est, max_depth=10, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f'sklearn RF (n={n_est}): accuracy={acc:.4f}')

## Gradient Boosting Classifier

In [ ]:
for lr in [0.01, 0.1, 0.5]:
    clf = GradientBoostingClassifier(n_estimators=100, learning_rate=lr,
                                     max_depth=3, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f'sklearn GBT (lr={lr}): accuracy={acc:.4f}')

## Random Forest Regressor

In [ ]:
X_reg, y_reg = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_train_r, X_test_r = X_reg[:400], X_reg[400:]
y_train_r, y_test_r = y_reg[:400], y_reg[400:]

for n_est in [10, 50, 100]:
    reg = RandomForestRegressor(n_estimators=n_est, max_depth=10, random_state=42)
    reg.fit(X_train_r, y_train_r)
    mse = mean_squared_error(y_test_r, reg.predict(X_test_r))
    print(f'sklearn RF Regressor (n={n_est}): MSE={mse:.2f}')

## Gradient Boosting Regressor

In [ ]:
for lr in [0.01, 0.1, 0.5]:
    reg = GradientBoostingRegressor(n_estimators=100, learning_rate=lr,
                                    max_depth=3, random_state=42)
    reg.fit(X_train_r, y_train_r)
    mse = mean_squared_error(y_test_r, reg.predict(X_test_r))
    print(f'sklearn GBT Regressor (lr={lr}): MSE={mse:.2f}')

## Feature Importance Comparison

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

gbt = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbt.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, name in [(axes[0], rf, 'Random Forest'), (axes[1], gbt, 'Gradient Boosting')]:
    imp = model.feature_importances_
    ax.barh(range(len(imp)), imp)
    ax.set_yticks(range(len(imp)))
    ax.set_yticklabels([f'Feature {i}' for i in range(len(imp))])
    ax.set_title(f'{name} — Feature Importances')
plt.tight_layout()
plt.show()

## RustML Comparison (via PyO3 bindings)

Build the wheel first: `cd crates/rustml-python && maturin develop --release`

In [ ]:
try:
    import rustml_python as rml

    rf_rml = rml.RandomForestClassifier(n_estimators=100, max_depth=10, seed=42)
    rf_rml.fit(X_train, y_train)
    preds_rml = rf_rml.predict(X_test)
    acc_rml = rml.accuracy_score(y_test, preds_rml)
    print(f'rustml RF Classifier: accuracy={acc_rml:.4f}')

    gbt_rml = rml.GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, seed=42)
    gbt_rml.fit(X_train, y_train)
    preds_gbt = gbt_rml.predict(X_test)
    acc_gbt = rml.accuracy_score(y_test, preds_gbt)
    print(f'rustml GBT Classifier: accuracy={acc_gbt:.4f}')
except ImportError:
    print('rustml_python not installed — run: cd crates/rustml-python && maturin develop --release')